# Данные Авито (СПб) — просмотр для команды

**Где идёт сбор.** Сбор работает на компьютере с Windows: сборщик подключается к отдельному окну Chrome, поэтому запросы идут с российского IP. Из Colab Авито блокирует запросы — это проверено. Как запустить сбор, написано в README репозитория, раздел «Сбор на Windows».

**Что делает этот ноутбук.** Сборщик складывает данные в общую папку на Google Drive: базу, фото, отчёты, журналы. Ноутбук берёт снимок базы и показывает отчёт, таблицы, фото и ошибки, а также делает выгрузку для эконометристов.

В базу ноутбук **ничего не пишет**, поэтому его могут одновременно открывать все участники.

Порядок работы: ячейки 1–3, затем любые нужные.

In [ ]:
#@title 1. Параметры { display-mode: "form" }
#@markdown Папка проекта на Google Drive — та же, что выбрана при настройке сборщика. Если её вам расшарили, добавьте её в «Мой диск» через «Добавить ярлык».
PROJECT_DIR = "/content/drive/MyDrive/Scraper_avito"  #@param {type:"string"}
#@markdown Репозиторий с кодом и ветка (пусто — основная).
REPO_URL = "https://github.com/neznanowt/avito-scrapper-hse"  #@param {type:"string"}
REPO_BRANCH = ""  #@param {type:"string"}

In [ ]:
#@title 2. Подключить Drive и взять код (несколько секунд) { display-mode: "form" }
import os, shutil, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")  # «Drive already mounted» при повторном запуске — это нормально

PROJECT = Path(PROJECT_DIR)
DATA = PROJECT / "data"
CODE = Path("/content/realty-scraper") if IN_COLAB else Path(os.environ.get("REALTY_CODE_DIR", "realty-scraper")).resolve()


def git(*args):
    r = subprocess.run(["git", *map(str, args)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip() or r.stdout.strip())
    return r.stdout.strip()


if (CODE / ".git").exists():
    git("-C", CODE, "pull", "-q")
else:
    shutil.rmtree(CODE, ignore_errors=True)
    branch = ["-b", REPO_BRANCH.strip()] if REPO_BRANCH.strip() else []
    git("clone", "-q", *branch, REPO_URL.strip(), CODE)

# свежая версия кода, если ячейку запускают повторно; браузер и пакеты сбора здесь не нужны
for m in [m for m in sys.modules if m.startswith("realty_scraper")]:
    del sys.modules[m]
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
print("Код:", git("-C", CODE, "log", "-1", "--format=%h · %s"))
print("Данные:", DATA, "— найдены" if DATA.exists() else "— папки пока нет (сбор ещё не запускали?)")

In [ ]:
#@title 3. Снимок базы { display-mode: "form" }
#@markdown Копирует текущую базу с Drive в сессию. Запустите ещё раз, чтобы увидеть свежие данные после нового сбора.
import sqlite3
import pandas as pd
from IPython.display import HTML, Markdown, display
from realty_scraper.notebook import snapshot_db

DB = snapshot_db(DATA / "realty.sqlite", Path("/content/realty_snapshot.sqlite") if IN_COLAB else PROJECT / "_snapshot.sqlite")
con = sqlite3.connect(DB)
summary = pd.read_sql("""
    SELECT COUNT(*) AS объявлений,
           SUM(last_parsed_at IS NOT NULL) AS разобрано_карточек,
           SUM(status = 'active') AS активных,
           SUM(status IN ('closed', 'removed')) AS снятых
    FROM listings""", con)
photos = pd.read_sql("SELECT COUNT(*) AS фото, SUM(downloaded_at IS NOT NULL) AS скачано FROM photos WHERE is_current = 1", con)
display(pd.concat([summary, photos], axis=1))
print("Последние запуски сбора:")
display(pd.read_sql("""SELECT run_id AS №, mode AS режим, substr(started_at, 1, 16) AS начало,
                              substr(finished_at, 12, 5) AS конец, status AS статус
                       FROM runs ORDER BY run_id DESC LIMIT 8""", con))

## 4. Отчёт по всей базе
Заполненность полей, распределение поля «ремонт», фото на объявление, скорость сбора и прогноз объёма.

In [ ]:
from realty_scraper.report import full_report

con.row_factory = sqlite3.Row
display(Markdown(full_report(con, "avito")))
con.row_factory = None

## 5. Таблицы

In [ ]:
from realty_scraper.notebook import fill_rates, listings_df, renovation_overview

df = listings_df(DB)
display(fill_rates(df))
display(renovation_overview(df))

In [ ]:
#@title Цена за м² по полю «ремонт» { display-mode: "form" }
import matplotlib.pyplot as plt

parsed = df[df["price_per_m2"].notna() & df["renovation"].notna()]
if len(parsed):
    order = parsed.groupby("renovation")["price_per_m2"].median().sort_values().index.tolist()
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.boxplot([parsed.loc[parsed["renovation"] == r, "price_per_m2"] for r in order], showfliers=False)
    ax.set_xticks(range(1, len(order) + 1), order, rotation=15)
    ax.set_title("Цена предложения за м², ₽ (без выбросов)")
    plt.show()
else:
    print("Пока нет разобранных карточек с ценой и полем «ремонт».")

## 6. Фото объявления

In [ ]:
#@title 6. Фото объявления { display-mode: "form" }
#@markdown Пусто — случайное объявление с фото; запустите ещё раз, чтобы посмотреть другое.
LISTING_ID = ""  #@param {type:"string"}
from realty_scraper.notebook import photo_grid_html
display(HTML(photo_grid_html(DB, DATA, LISTING_ID or None, n=24)))

## 7. Ошибки сбора
Последние ошибки из журнала: блокировки, непонятные страницы, ошибки разбора. Полезно, если сбор на компьютере идёт не так.

In [ ]:
display(pd.read_sql("""SELECT substr(ts, 1, 16) AS время, stage AS этап, kind AS вид, message AS сообщение
                       FROM errors ORDER BY id DESC LIMIT 20""", sqlite3.connect(DB)))

## 8. Выгрузка для эконометристов
`listings.csv` и `photos.csv` в папке `data/export/` на Drive. Кодировка UTF-8, открываются в Excel; описание полей — `CODEBOOK.md` в репозитории.

In [ ]:
from realty_scraper.db import export_tables

for path, n in export_tables(DB, DATA / "export"):
    print(f"{path}: {n} строк")